In [11]:
import asyncio
import os

from dotenv import load_dotenv
from huggingface_hub import AsyncInferenceClient

# MWE: .env im Projektordner laden, dann Token direkt auslesen.
load_dotenv()

TOKEN = os.getenv("HF_TOKEN") or os.getenv("HF_API_KEY")
if not TOKEN:
    raise RuntimeError(
        "Kein Hugging Face Token gefunden. Setze HF_TOKEN oder HF_API_KEY in .env."
    )

client = AsyncInferenceClient(api_key=TOKEN)

# Dieses Chat-Modell wurde mit dem aktuellen Token erfolgreich verifiziert.
MODEL_CHAT = "unsloth/Qwen2.5-72B-Instruct:featherless-ai"
MODEL_EMBED = "Qwen/Qwen3-Embedding-8B"


def log_usage(label, response):
    usage = getattr(response, "usage", None)
    if usage is None:
        print(f"{label} usage: n/a")
        return
    print(
        f"{label} usage: prompt={usage.prompt_tokens}, "
        f"completion={usage.completion_tokens}, total={usage.total_tokens}"
    )


async def embed(texts: list[str]):
    return await client.feature_extraction(texts, model=MODEL_EMBED)


async def translate(texts: list[str], src="de", tgt="en", max_concurrent=5):
    sem = asyncio.Semaphore(max_concurrent)

    async def _one(text):
        async with sem:
            response = await client.chat_completion(
                model=MODEL_CHAT,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            f"Translate from {src} to {tgt}. "
                            f"Output ONLY the translation:\n\n{text}"
                        ),
                    }
                ],
                max_tokens=512,
            )
            log_usage("translation", response)
            return response.choices[0].message.content

    return await asyncio.gather(*[_one(text) for text in texts])


async def label_topics(keyword_lists: list[str], max_concurrent=5):
    sem = asyncio.Semaphore(max_concurrent)

    async def _one(keywords):
        async with sem:
            response = await client.chat_completion(
                model=MODEL_CHAT,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            f"Topic keywords: {keywords}\n"
                            "Give a short label (max 5 words). Output ONLY the label."
                        ),
                    }
                ],
                max_tokens=30,
            )
            log_usage("label", response)
            return response.choices[0].message.content

    return await asyncio.gather(*[_one(keywords) for keywords in keyword_lists])


async def main():
    texts = [
        "Die Gravitationsphysik in der DDR entwickelte sich unter besonderen Bedingungen.",
        "Treder formulierte seine eigene Gravitationstheorie.",
        "Die Akademie der Wissenschaften war zentral organisiert.",
    ]

    embeddings = await embed(texts)
    print(f"Embeddings: {embeddings.shape}")

    translations = await translate(texts)
    print("\nTranslations:")
    for item in translations:
        print(f"- {item}")

    labels = await label_topics([
        "gravitation, Relativitaet, DDR, Treder, Feldtheorie",
        "Fleck, Denkstil, Denkkollektiv, Wissenssoziologie",
    ])
    print("\nLabels:")
    for item in labels:
        print(f"- {item}")

await main()

Embeddings: (3, 4096)
translation usage: prompt=54, completion=8, total=62
translation usage: prompt=61, completion=13, total=74
translation usage: prompt=55, completion=10, total=65

Translations:
- Gravitational physics in the GDR developed under special conditions.
- Treder formulated his own theory of gravitation.
- The Academy of Sciences was centrally organized.
label usage: prompt=68, completion=6, total=74
label usage: prompt=64, completion=6, total=70

Labels:
- German Relativity and Gravity Theory
- Sociology of Knowledge Concepts
